# Lesson 22 Lab — PyTorch Integration with torch.compile

**Puzzle:** When graph capture, Inductor, generated kernels, and cold start change together, which observation tells you whether the kernel, layout, toolchain, or hardware boundary is responsible?

This notebook retains one complete RTX 5090 execution.


## Why this matters

This lab isolates graph capture, Inductor, generated kernels, and cold start and keeps its comparison path explicit.


## 0. Predict before running

Predict correctness, warm latency ordering, and the first boundary case. Write what would disprove each prediction.


## 1. Theory and mechanism

torch.compile captures a PyTorch graph and lets Inductor generate fused code, often using Triton on CUDA. This route can remove operator boundaries without maintaining a manual kernel, but compilation, graph breaks, guards, and dynamic shapes become part of the system.


## 2. Trace the mechanism

```mermaid
flowchart LR
  A["Frozen input + contract"] --> B["graph capture, Inductor, generated kernels, and cold start"]
  B --> C["Triton candidate"]
  B --> D["CUDA / library control"]
  C --> E["correctness + samples"]
  D --> E
  E --> F["bounded decision"]
```


## 3. Inspect the comparison boundary

Baseline: named PyTorch CUDA/library or standard-grid path. Candidate: optimized framework path described in the experiment.

Comparing compiled cold start with eager warm time, or attributing all compiled speedup to one generated kernel, produces an invalid conclusion.


## 4. Inspect the execution environment

The next cell asserts CUDA and records GPU, target, PyTorch, CUDA runtime, Triton, Python, and seed.


In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT / "scripts"))
from chapter05_runtime import environment, run_lesson

LESSON_NO = 22
LESSON_TITLE = 'PyTorch Integration with torch.compile'
ENV = environment(LESSON_NO)
print(json.dumps(ENV, indent=2, ensure_ascii=False))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "triton": "3.7.1",
  "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
  "python": "3.12.3",
  "seed": 20260835
}


## 5. Freeze the experiment

**Experiment:** Compile a pointwise PyTorch subgraph and record cold execution, compiled warm samples, eager warm samples, and error.

Inputs, output contract, timer, and target stay fixed across compared paths.


## 6. Inspect and execute the reviewed code

The next cell calls the shared reviewed kernel source, retains full samples in `metrics`, checks maximum error, and prints the bounded analysis.


In [2]:
metrics, analysis_en, analysis_zh = run_lesson(LESSON_NO)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print(analysis_en)


{
  "primary": 1013.719717040658,
  "secondary": 0.05571199953556061,
  "max_abs_error": 1.7881393432617188e-07,
  "passed": true,
  "details": {
    "eager_median_ms": 0.02611199952661991,
    "compiled_samples_ms": [
      0.06864000111818314,
      0.06505600363016129,
      0.0575999990105629,
      0.05548800155520439,
      0.05737600103020668,
      0.06083200126886368,
      0.05571199953556061,
      0.05462399870157242,
      0.05596800148487091,
      0.05363199859857559,
      0.053727999329566956,
      0.05488000065088272,
      0.054816000163555145,
      0.07340800017118454,
      0.05392000079154968
    ],
    "eager_samples_ms": [
      0.03097599931061268,
      0.026048000901937485,
      0.024831999093294144,
      0.026240000501275063,
      0.032607998698949814,
      0.026335999369621277,
      0.024927999824285507,
      0.024671999737620354,
      0.026623999699950218,
      0.02611199952661991,
      0.026464000344276428,
      0.02521600015461445,
      0.02

## 7. Read the retained RTX 5090 result

**Environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Triton 3.7.1; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Compiled cold call | 1013.7197 ms |
| Compiled warm median | 0.0557 ms |
| Maximum absolute error | 1.788e-07 |
| Acceptance gate | true |


## 8. Explain without overclaiming

torch.compile cold execution took 1013.72 ms; its warm median was 0.0557 ms versus 0.0261 ms eager. Compile cost and steady state stay separate.

A named Triton or PyTorch CUDA path executed on the recorded GPU. The result applies to the printed shape, dtype, implementation, and software stack; internal hardware causes require profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, full metrics, bilingual analysis, evidence label, and bounded conclusion.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": LESSON_NO,
    "title": LESSON_TITLE,
    "environment": ENV,
    "evidence_label": 'native-backend',
    "metrics": metrics,
    "analysis_en": analysis_en,
    "analysis_zh": analysis_zh,
    "conclusion": 'Try compiler fusion before owning a manual kernel when the graph is stable and generated performance meets the gate.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 22,
  "title": "PyTorch Integration with torch.compile",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "triton": "3.7.1",
    "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
    "python": "3.12.3",
    "seed": 20260835
  },
  "evidence_label": "native-backend",
  "metrics": {
    "primary": 1013.719717040658,
    "secondary": 0.05571199953556061,
    "max_abs_error": 1.7881393432617188e-07,
    "passed": true,
    "details": {
      "eager_median_ms": 0.02611199952661991,
      "compiled_samples_ms": [
        0.06864000111818314,
        0.06505600363016129,
        0.0575999990105629,
        0.05548800155520439,
        0.05737600103020668,
        0.06083200126886368,
        0.05571199953556061,
        0.05462399870157242,
        0.05596800148487091,
        0.05363199859857559,
        0.053727999329566956,
        0.05488000065088272,
  

## 10. Make the bounded decision

> Try compiler fusion before owning a manual kernel when the graph is stable and generated performance meets the gate.

**Failure analysis:** Comparing compiled cold start with eager warm time, or attributing all compiled speedup to one generated kernel, produces an invalid conclusion.


## 11. Extend and review

Add an awkward shape and non-contiguous layout. Stop on correctness failure. See `README.md` for references and the full review checklist.
